<a href="https://colab.research.google.com/github/75hbchy4kr-wq/AI/blob/main/My_First_AI_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)
print("My AI development environment is ready!")

PyTorch version: 2.14.1+cpu
My AI development environment is ready!


In [17]:
profile = {
    "name": "Irene",
    "birthday": "12th December",
    "favorite_subject": "precal",
    "hobby": "music",
    "future_career": "bus driver",
    "favorite_color": "green",
    "favorite_food": "pasta",
    "favorite_sport": "rugby"
}

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.14.1+cpu
GPU available: False


In [19]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Training device:", device)

Training device: cpu


In [20]:
!pip install -q fastapi uvicorn requests
!mkdir -p data

In [21]:
import random
random.seed(0)

facts = {
 "name": ("Your name is Irene.", "Yes, your name is Irene.", [
   "What is my name?", "Who am I?", "Tell me my name.", "What am I called?", "Say my name.",
   "What's my name?", "Can you tell me my name?", "What name do I have?", "Remind me what my name is.",
   "Who is the person talking to you?", "I forgot my name, what is it?", "My name is what?"]),
 "birthday": ("Your birthday is December 12.", "Yes, your birthday is December 12.", [
   "When is my birthday?", "What is my birthday?", "Tell me my birthday.", "What day was I born?",
   "What date is my birthday?", "When was I born?", "What's my birthday?", "Which day is my birthday?",
   "Remind me when my birthday is.", "On what date do I celebrate my birthday?", "My birthday is on which day?"]),
 "height": ("Your height is 180 cm.", "Yes, you are 180 cm tall.", [
   "What is my height?", "How tall am I?", "Tell me my height.", "What is my height in centimeters?",
   "How many cm tall am I?", "What's my height?", "Can you tell me how tall I am?", "How big am I?",
   "Remind me how tall I am.", "What is my body height?", "How many centimeters tall am I?", "Give me my height."]),
 "weight": ("Your weight is 100 kg.", "Yes, you weigh 100 kg.", [
   "What is my weight?", "How much do I weigh?", "Tell me my weight.", "What is my weight in kilograms?",
   "How many kg do I weigh?", "What's my weight?", "Can you tell me how heavy I am?", "How heavy am I?",
   "Remind me how much I weigh.", "What is my body weight?", "How many kilograms do I weigh?", "Give me my weight."]),
 "shoe": ("Your shoe size is 18.", "Yes, your shoe size is 18.", [
   "What is my shoe size?", "What size shoes do I wear?", "Tell me my shoe size.", "What shoe size am I?",
   "Which size of shoes fits me?", "What's my shoe size?", "Can you tell me my shoe size?", "What size are my shoes?",
   "Remind me what size shoes I wear.", "How big are my feet?", "Which shoe size should I buy?", "Give me my shoe size."]),
 "subject": ("Your favorite subject is Computer Science.", "Yes, your favorite subject is Computer Science.", [
   "What is my favorite subject?", "Which subject do I like most?", "Tell me my favorite subject.",
   "What subject do I enjoy the most?", "What is my best subject?", "What's my favorite subject?",
   "Can you tell me which subject I love?", "Which class do I like best?", "Remind me what my favorite subject is.",
   "What do I like studying most?", "Which school subject is my favorite?", "Give me my favorite subject."]),
 "hobby": ("Your hobby is sleeping.", "Yes, your hobby is sleeping.", [
   "What is my hobby?", "What do I do for fun?", "Tell me my hobby.", "What do I like to do in my free time?",
   "Which activity do I enjoy?", "What's my hobby?", "Can you tell me what I do in my spare time?",
   "What do I enjoy doing?", "Remind me what my hobby is.", "How do I spend my free time?",
   "What is my favorite activity?", "Give me my hobby."]),
 "goal": ("Your future goal is to get 100 in Computer Science.", "Yes, your future goal is to get 100 in Computer Science.", [
   "What is my future goal?", "What do I want to achieve?", "Tell me my future goal.",
   "What score do I want in the future?", "What is my dream grade?", "What's my goal?",
   "Can you tell me what I want to achieve?", "What am I working toward?", "Remind me what my goal is.",
   "What is my target grade?", "What do I hope to get in the future?", "Give me my goal."]),
}
wrappers = ["{q}", "Please {ql}", "Hey, {ql}", "Quick question: {ql}", "{q} Please.", "I forgot. {q}"]

lines = []
for key, (ans, yes, qs) in facts.items():
    for q in qs:
        for w in wrappers:
            ql = q[0].lower() + q[1:]
            lines.append(w.format(q=q, ql=ql) + "|" + ans)
    lines.append("Do you know my " + key + "?|" + yes)

# things the model should NOT know
unknown = ["What is my favorite color?", "What is my phone number?", "Where do I live?",
           "What is my favorite food?", "How old am I?", "What is my mother's name?"]
for q in unknown:
    for w in wrappers:
        lines.append(w.format(q=q, ql=q[0].lower() + q[1:]) + "|I do not know that.")

random.shuffle(lines)
open("data/qa.txt", "w").write("\n".join(lines))
print(len(lines), "training examples")

614 training examples


In [22]:
%%writefile data/test_new.txt
Could you remind me how tall I am?|You are 180 cm tall.
Please tell me what my shoe size is.|Your shoe size is 18.
Which month is my birthday in?|Your birthday is December 12.
What do I weigh?|You weigh 100 kg.
What is my favourite school subject?|Your favorite subject is Computer Science.
What do I want to accomplish?|You want to get 100 in Computer Science.

Overwriting data/test_new.txt


In [23]:
%%writefile common.py
import torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

PAD, SOS, EOS = 0, 1, 2

class Vocab:
    def __init__(self, chars):
        self.chars = list(chars)
        self.stoi = {c: i + 3 for i, c in enumerate(self.chars)}
        self.itos = {i + 3: c for i, c in enumerate(self.chars)}
    @classmethod
    def build(cls, texts):
        return cls(sorted(set("".join(texts))))
    def encode(self, s):
        return [self.stoi[c] for c in s if c in self.stoi]
    def decode(self, ids):
        return "".join(self.itos.get(i, "") for i in ids)
    def __len__(self):
        return len(self.chars) + 3

def load_pairs(path):
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if "|" in line:
                q, a = line.rstrip("\n").split("|", 1)
                pairs.append((q.strip().lower(), a.strip()))
    return pairs

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb=64, hid=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD)
        self.enc = nn.GRU(emb, hid, batch_first=True)
        self.dec = nn.GRU(emb, hid, batch_first=True)
        self.out = nn.Linear(hid, vocab_size)

    def encode(self, src, lengths):
        packed = pack_padded_sequence(self.emb(src), lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, h = self.enc(packed)
        return h

    def forward(self, src, lengths, tgt_in):
        h = self.encode(src, lengths)
        o, _ = self.dec(self.emb(tgt_in), h)
        return self.out(o)

    @torch.no_grad()
    def generate(self, src, lengths, max_len=100):
        h = self.encode(src, lengths)
        tok = torch.full((1, 1), SOS, dtype=torch.long)
        out = []
        for _ in range(max_len):
            o, h = self.dec(self.emb(tok), h)
            tok = self.out(o[:, -1]).argmax(-1, keepdim=True)
            if tok.item() == EOS:
                break
            out.append(tok.item())
        return out

def predict(model, vocab, question):
    model.eval()
    ids = vocab.encode(question.strip().lower()) or [3]
    return vocab.decode(model.generate(torch.tensor([ids]), torch.tensor([len(ids)])))

Overwriting common.py


In [24]:
%%writefile train.py
import random, torch, torch.nn as nn
from common import *

EPOCHS, LR, BATCH = 300, 3e-3, 16
random.seed(0); torch.manual_seed(0)

pairs = load_pairs("data/qa.txt")
vocab = Vocab.build([q + a for q, a in pairs])
print(f"{len(pairs)} training examples, vocab size {len(vocab)}")

def batchify(batch):
    srcs = [vocab.encode(q) for q, _ in batch]
    tgts = [[SOS] + vocab.encode(a) + [EOS] for _, a in batch]
    pad = lambda seqs: torch.tensor([s + [PAD] * (max(map(len, seqs)) - len(s)) for s in seqs])
    return pad(srcs), torch.tensor([len(s) for s in srcs]), pad(tgts)

model = Seq2Seq(len(vocab))
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
n_batches = (len(pairs) + BATCH - 1) // BATCH

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(pairs)
    total = 0
    for i in range(0, len(pairs), BATCH):
        src, lengths, tgt = batchify(pairs[i:i + BATCH])
        logits = model(src, lengths, tgt[:, :-1])
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | training loss {total / n_batches:.4f}")

torch.save({"state": model.state_dict(), "chars": vocab.chars}, "model.pt")
print("Saved model.pt")

Overwriting train.py


In [25]:
!python train.py

614 training examples, vocab size 43
Epoch   1 | training loss 1.3639
Epoch  10 | training loss 0.0723
Epoch  20 | training loss 0.0231
Epoch  30 | training loss 0.0052
Epoch  40 | training loss 0.0012
Epoch  50 | training loss 0.0002
Epoch  60 | training loss 0.0001
Epoch  70 | training loss 0.0001
Epoch  80 | training loss 0.0000
Epoch  90 | training loss 0.0000
Epoch 100 | training loss 0.0000
Epoch 110 | training loss 0.0000
Epoch 120 | training loss 0.0000
Epoch 130 | training loss 0.0000
Epoch 140 | training loss 0.0000
Epoch 150 | training loss 0.0000
Epoch 160 | training loss 0.0000
Epoch 170 | training loss 0.0000
Epoch 180 | training loss 0.0000
Epoch 190 | training loss 0.0000
Epoch 200 | training loss 0.0000
Epoch 210 | training loss 0.0000
Epoch 220 | training loss 0.0000
Epoch 230 | training loss 0.0000
Epoch 240 | training loss 0.0000
Epoch 250 | training loss 0.0000
Epoch 260 | training loss 0.0000
Epoch 270 | training loss 0.0000
Epoch 280 | training loss 0.0000
Epoch 

In [26]:
import torch
from common import *

ckpt = torch.load("model.pt")
vocab = Vocab(ckpt["chars"])
model = Seq2Seq(len(vocab)); model.load_state_dict(ckpt["state"])

for title, path in [("TEST A - seen in training", "data/qa.txt"),
                    ("TEST B - NEW questions", "data/test_new.txt")]:
    pairs = load_pairs(path)
    if path.endswith("qa.txt"): pairs = pairs[::6]
    print(f"\n{title}")
    correct = 0
    for q, expected in pairs:
        got = predict(model, vocab, q)
        ok = got == expected; correct += ok
        print(f"Q: {q}\n  expected: {expected}\n  model:    {got}  {'OK' if ok else 'WRONG'}")
    print(f"Exact match: {correct}/{len(pairs)}")


TEST A - seen in training
Q: hey, what's my goal?
  expected: Your future goal is to get 100 in Computer Science.
  model:    Your future goal is to get 100 in Computer Science.  OK
Q: i forgot. what is my body height?
  expected: Your height is 180 cm.
  model:    Your height is 180 cm.  OK
Q: tell me my height. please.
  expected: Your height is 180 cm.
  model:    Your height is 180 cm.  OK
Q: what's my hobby?
  expected: Your hobby is sleeping.
  model:    Your hobby is sleeping.  OK
Q: i forgot. which size of shoes fits me?
  expected: Your shoe size is 18.
  model:    Your shoe size is 18.  OK
Q: please what do i want to achieve?
  expected: Your future goal is to get 100 in Computer Science.
  model:    Your future goal is to get 100 in Computer Science.  OK
Q: what do i enjoy doing? please.
  expected: Your hobby is sleeping.
  model:    Your hobby is sleeping.  OK
Q: hey, give me my shoe size.
  expected: Your shoe size is 18.
  model:    Your shoe size is 18.  OK
Q: what is 

In [27]:
%%writefile api.py
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from common import *

app = FastAPI(title="My Own AI API")
ckpt = torch.load("model.pt")
vocab = Vocab(ckpt["chars"])
model = Seq2Seq(len(vocab)); model.load_state_dict(ckpt["state"]); model.eval()

class Question(BaseModel):
    question: str

@app.post("/ask")
def ask(body: Question):
    return {"answer": predict(model, vocab, body.question)}

Overwriting api.py


In [28]:
!nohup uvicorn api:app --port 8000 > api.log 2>&1 &
import time; time.sleep(8)

In [29]:
import requests
r = requests.post("http://127.0.0.1:8000/ask", json={"question": "What is my shoe size?"})
print(r.json())

{'answer': 'Your shoe size is 18.'}


In [30]:
%%writefile app.py
import requests, streamlit as st

st.title("Irene's Personal AI Assistant")
q = st.text_input("Ask me a question about yourself:")
if st.button("Ask") and q:
    r = requests.post("http://127.0.0.1:8000/ask", json={"question": q}, timeout=10)
    st.write(r.json()["answer"])

Overwriting app.py


In [32]:
import requests
r = requests.post("http://127.0.0.1:8000/ask", json={"question": "do you know what'a my favorite subject??"})
print(r.json())

{'answer': 'Yes, your favorite subject is Computer Science.'}


In [33]:
import streamlit as st
import requests

st.title("Irene's Personal AI Assistant")
question = st.text_input("Ask me a question:")

if st.button("Ask"):
    r = requests.post("http://127.0.0.1:8000/ask", json={"question": question})
    st.write(r.json()["answer"])

ModuleNotFoundError: No module named 'streamlit'